## This is my attempt to cull together some key ideas and computational tools I use consistently for data analysis

# Very exhaustive imports

In [1]:
#Standards

import numpy as np 
import matplotlib.pyplot as plt
import sys
import sympy as sym
import seaborn
seaborn.set_theme()
from typing import Optional
from mpl_toolkits import mplot3d

#If fitting or using gaussians

from scipy.ndimage import gaussian_filter 
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.signal import find_peaks

# Miscellaneous 

In [ ]:
#2D Color Plotting

plt.figure(figsize=(10,8))
plt.imshow(np.log(dev), cmap = 'RdPu', extent =[0, 5000, 5000, 0], interpolation='nearest')
plt.xlabel('Level Density 1')
plt.ylabel('Level Density 2')
plt.title('Angle 1')
plt.colorbar()
plt.show()

In [ ]:
# Load Text from CSV

file = r""

dat = np.genfromtxt(file, delimiter=",", names=True, dtype=None, encoding="utf-8")

# Workflow for Collinear Laser Spectroscopy 

In [ ]:
#Gaussian fitting/plotting function for spectroscopy

def gauss(x, A, x0, sig, y0):
    return A * np.exp(-(x - x0) ** 2 / (2 * sig ** 2)) + y0

def _estimate_window_from_fwhm(x, y_smooth, i_peak, frac=0.5):
    """
    Estimate a half-width window around the peak using the smoothed trace.
    Finds the left/right crossing of y_peak*frac (default half-max).
    Returns half_width in x units. Falls back to ~10 bins if crossings fail.
    """
    y_peak = y_smooth[i_peak]
    if not np.isfinite(y_peak) or y_peak <= 0:
        return None

    level = y_peak * frac

    # search left
    i_left = i_peak
    while i_left > 0 and y_smooth[i_left] > level:
        i_left -= 1

    # search right
    i_right = i_peak
    n = len(y_smooth)
    while i_right < n - 1 and y_smooth[i_right] > level:
        i_right += 1

    # If we didn't find a crossing on either side, return None
    if i_left == 0 or i_right == n - 1:
        return None

    x_left = x[i_left]
    x_right = x[i_right]
    fwhm = x_right - x_left
    if not np.isfinite(fwhm) or fwhm <= 0:
        return None

    return 0.5 * fwhm  # half-width at half-max


def plot_hist_spec_gaussfit_auto_window(
    file,
    bins,
    title,
    xlabel=r'Wavenumber $(\mathrm{cm}^{-1})$',
    ylabel='Counts',
    color="C0",
    wn_col="wn_1",
    smooth_sigma=1.0,          # smoothing only for peak-finding/windowing
    window_scale=3.0,          # fit window half-width = window_scale * (FWHM/2)
    min_window_bins=12,        # fallback minimum half-window in bins (each side)
    maxfev=20000,
    plot_window_scale = 4.0,
):
    """
    Histogram -> smooth for peak+window estimation -> automatic fit window from FWHM
    -> Gaussian fit to RAW counts in that window (Poisson weights)
    -> plots both peak estimates (argmax of smoothed, and Gaussian-fit center).

    Returns dict of results.
    """

    # --- load ---
    dat = np.genfromtxt(
        file,
        delimiter=",",
        names=True,
        dtype=None,
        encoding="utf-8",
    )
    if wn_col not in dat.dtype.names:
        raise KeyError(f"Column '{wn_col}' not found. Available: {dat.dtype.names}")

    x = np.array(dat[wn_col], dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        raise ValueError(f"No finite data found in column '{wn_col}'.")

    # --- histogram ---
    counts, bin_edges = np.histogram(x, bins=bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    dx = np.median(np.diff(bin_centers))

    # --- smooth for peak estimate / windowing ---
    smooth_counts = gaussian_filter1d(counts, sigma=smooth_sigma) if smooth_sigma else counts

    i_peak = int(np.argmax(smooth_counts))
    wn_peak = float(bin_centers[i_peak])

    # --- automatic window half-width ---
    half_w = _estimate_window_from_fwhm(bin_centers, smooth_counts, i_peak, frac=0.5)

    if half_w is None:
        # fallback: use a minimum number of bins around the peak
        half_w = float(min_window_bins * dx)
    else:
        # scale it up to include enough of the lineshape for a stable fit
        half_w = float(window_scale * half_w)

    # guardrail: ensure at least min_window_bins worth of half-width
    half_w = max(half_w, float(min_window_bins * dx))

    # --- fit region ---
    mask = (bin_centers >= wn_peak - half_w) & (bin_centers <= wn_peak + half_w)
    x_fit = bin_centers[mask]
    y_fit = counts[mask]  # RAW counts for fitting

    if x_fit.size < 6:
        raise ValueError(
            f"Automatic window produced too few points ({x_fit.size}). "
            f"Try increasing 'bins' or lowering 'min_window_bins'."
        )

    # --- Poisson-ish weights ---
    sigma_y = np.sqrt(np.maximum(y_fit, 1.0))

    # --- initial guesses ---
    A0 = float(y_fit.max() - np.median(y_fit))
    x00 = float(x_fit[np.argmax(y_fit)])
    sig0 = float(max(half_w / 6, 1e-3))  # rough: sigma smaller than window
    y00 = float(np.median(y_fit))
    p0 = [A0, x00, sig0, y00]

    # --- bounds ---
    lower = [0.0, wn_peak - half_w, 1e-4, -np.inf]
    upper = [np.inf, wn_peak + half_w, half_w, np.inf]

    # --- fit ---
    popt, pcov = curve_fit(
        gauss,
        x_fit,
        y_fit,
        p0=p0,
        bounds=(lower, upper),
        sigma=sigma_y,
        absolute_sigma=True,
        maxfev=maxfev,
    )

    A, x0, sig, y0 = popt
    x0_err = float(np.sqrt(np.diag(pcov))[1]) if np.all(np.isfinite(pcov)) else np.nan

    # --- model curve ---
    x_model = np.linspace(x_fit.min(), x_fit.max(), 1200)
    y_model = gauss(x_model, *popt)

    # --- plot ---
    fig, ax = plt.subplots(figsize=(15, 10))

    ax.step(bin_centers, counts, where="mid", color=color, alpha=0.30)
    ax.step(bin_centers, smooth_counts, where="mid", color=color, alpha=0.95, label="Smoothed Histogram")

    # show auto fit window
    ax.axvspan(wn_peak - half_w, wn_peak + half_w, alpha=0.10, label="Auto fit window")

    # fit data + fit curve
    ax.errorbar(x_fit, y_fit, yerr=sigma_y, fmt="o", color=color, markersize=4, label="Fit data (raw counts)")
    ax.plot(x_model, y_model, color="C3", linewidth=2.5, label="Gaussian fit")

    # both peak markers
    ax.axvline(wn_peak, linestyle=":", color="k",
               label=rf"Argmax(smoothed) = {wn_peak:.6f} cm$^{{-1}}$")
    ax.axvline(x0, linestyle="--", color="k",
               label=rf"$x_0$ (fit) = {x0:.6f} $\pm$ {x0_err:.6f} cm$^{{-1}}$")

    ax.set_title(title, fontweight="bold", fontsize="x-large")
    ax.set_xlabel(xlabel, fontsize="x-large", fontweight="bold")
    ax.set_ylabel(ylabel, fontsize="x-large", fontweight="bold")
    ax.legend()
    
    plot_half_w = plot_window_scale * half_w
    plot_half_w = 4.0 * half_w

    x_min_data = bin_centers.min()
    x_max_data = bin_centers.max()

    x_left  = max(wn_peak - plot_half_w, x_min_data)
    x_right = min(wn_peak + plot_half_w, x_max_data)

    ax.set_xlim(x_left, x_right)
    fig.tight_layout()
    plt.show()

    return {
        "wn_peak_argmax_smoothed": wn_peak,
        "fit_center": float(x0),
        "fit_center_err": x0_err,
        "sigma": float(sig),
        "fwhm": float(2.355 * sig),
        "auto_half_window": float(half_w),
        "bin_centers": bin_centers,
        "counts": counts,
        "smooth_counts": smooth_counts,
        "x_fit": x_fit,
        "y_fit": y_fit,
        "popt": popt,
        "pcov": pcov}

In [ ]:
#Example plotting

res = plot_hist_spec_gaussfit_auto_window(
    file=r"C:\Users\j_hac\OneDrive\Desktop\Exotic Molecules lab\Sulfur\WithSpectro\scan_2026_01_09_18_55_59_fullscan_15mW.csv",
    bins=300,
    title=r"15 mW $^{32}\mathrm{S}$ Spectroscopic Resonance",
    color="g",
    wn_col="wn_1",
    smooth_sigma=1.0,
    window_scale=3.0)   # includes several linewidths for stable fits

print(f"Argmax center  = {res['wn_peak_argmax_smoothed']:.6f} cm^-1")
print(f"Fit center     = {res['fit_center']:.6f} ± {res['fit_center_err']:.6f} cm^-1")
print(f"FWHM (Gaussian)= {res['fwhm']:.6f} cm^-1")
print(f"Auto half-window used = {res['auto_half_window']:.6f} cm^-1")

In [ ]:
#Fitting multiple gaussians in same plot

import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import find_peaks
from scipy.optimize import minimize


# -------------------------
# Window estimation (gap-tolerant; no smoothing)
# -------------------------
def _estimate_window_from_fwhm_gap_tolerant(x, y, i_peak, frac=0.5, gap_bins=2):
    """
    Estimate a half-width around a peak using crossings at y_peak*frac (default half-max),
    but tolerate small gaps (e.g., zeros) inside the peak.

    Parameters
    ----------
    x : array
        Bin centers.
    y : array
        Histogram counts (RAW, not smoothed).
    i_peak : int
        Index of peak bin.
    frac : float
        Fraction of peak height to use (0.5 = half max).
    gap_bins : int
        How many consecutive bins below threshold are allowed before stopping.
        This prevents a few zero bins inside a peak from shrinking the window.

    Returns
    -------
    half_width : float or None
        Half-width in x units (i.e., 0.5*FWHM). Returns None if cannot estimate.
    """
    y_peak = y[i_peak]
    if not np.isfinite(y_peak) or y_peak <= 0:
        return None

    level = y_peak * frac
    n = len(y)

    # search left with tolerance for gaps below level
    i_left = i_peak
    gap = 0
    while i_left > 0:
        i_left -= 1
        if y[i_left] > level:
            gap = 0
        else:
            gap += 1
            if gap > gap_bins:
                # step back to last bin above level (approx)
                i_left = min(i_left + gap_bins, n - 1)
                break

    # search right with tolerance for gaps below level
    i_right = i_peak
    gap = 0
    while i_right < n - 1:
        i_right += 1
        if y[i_right] > level:
            gap = 0
        else:
            gap += 1
            if gap > gap_bins:
                i_right = max(i_right - gap_bins, 0)
                break

    if i_left <= 0 or i_right >= n - 1:
        return None

    fwhm = x[i_right] - x[i_left]
    if not np.isfinite(fwhm) or fwhm <= 0:
        return None

    return 0.5 * fwhm


# -------------------------
# Unbinned MLE helpers
# -------------------------
_SQRT2PI = np.sqrt(2.0 * np.pi)

def _normal_pdf(x, mu, sig):
    sig = np.maximum(sig, 1e-12)
    z = (x - mu) / sig
    return np.exp(-0.5 * z * z) / (sig * _SQRT2PI)

def _softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    ez = np.exp(z)
    return ez / np.sum(ez)

def _sigmoid(a):
    a = np.asarray(a, dtype=float)
    out = np.empty_like(a)
    pos = a >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-a[pos]))
    ea = np.exp(a[~pos])
    out[~pos] = ea / (1.0 + ea)
    return out


def plot_hist_spec_multigauss_auto_window(
    file,
    bins,
    title,
    xmin,
    xmax,
    xlabel=r"Wavenumber $(\mathrm{cm}^{-1})$",
    ylabel="Counts",
    color="C0",
    wn_col="wn_3",
    window_scale=3.0,
    min_window_bins=12,
    plot_window_scale=None,    # None => show full x-range
    # multi-peak controls:
    n_peaks=1,
    peak_guesses=None,         # list of approximate centers (cm^-1); overrides n_peaks autodetect
    peak_prominence=None,      # passed to find_peaks; if None, chosen heuristically
    peak_min_distance_bins=10, # separation in bins for find_peaks
    # fit-window expansion:
    fit_margin_bins=2.0,       # at least this many bins of extra margin
    fit_margin_frac=0.10,      # also expand by this fraction of union span
    # ---- Unbinned MLE controls ----
    use_unbinned_mle=True,
    mle_include_uniform_bg=True,
    mle_maxiter=2000,
    mle_print=False,
    show_fit_centers=False,
    # ---- NEW: handle zero-gaps in histogram peaks when estimating window ----
    fwhm_gap_bins=2,
):
    """
    Histogram (NO smoothing) -> choose peaks -> auto fit window via gap-tolerant FWHM
    -> UNBINNED MLE fit in that window (avoids zero-bin artifacts in fitting)
    -> plot histogram + components + total expected counts curve + argmax markers
    """

    # --- load ---
    #dat = np.genfromtxt(file, delimiter=",", names=True, dtype=None, encoding="utf-8")
    dat = file
    if wn_col not in dat.dtype.names:
        raise KeyError(f"Column '{wn_col}' not found. Available: {dat.dtype.names}")

    x_all = np.array(dat[wn_col], dtype=float)
    x_all = x_all[np.isfinite(x_all)]
    if x_all.size == 0:
        raise ValueError(f"No finite data found in column '{wn_col}'.")

    # --- histogram (for peak picking + plotting only) ---
    counts, bin_edges = np.histogram(x_all, bins=bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    if bin_centers.size < 2:
        raise ValueError("Too few bins to fit. Increase bins.")
    dx = float(np.median(np.diff(bin_centers)))

    # NO SMOOTHING
    y_for_peaks = counts

    # --- choose peak indices ---
    if peak_guesses is not None:
        peak_guesses = np.array(peak_guesses, dtype=float)
        peak_inds = [int(np.argmin(np.abs(bin_centers - g))) for g in peak_guesses]
        peak_inds = sorted(set(peak_inds))
        if len(peak_inds) == 0:
            peak_inds = [int(np.argmax(y_for_peaks))]
    else:
        if peak_prominence is None:
            m = float(np.max(y_for_peaks)) if np.max(y_for_peaks) > 0 else 1.0
            peak_prominence = 0.10 * m

        peaks, _props = find_peaks(
            y_for_peaks,
            prominence=float(peak_prominence),
            distance=int(peak_min_distance_bins),
        )

        if peaks.size == 0:
            peak_inds = [int(np.argmax(y_for_peaks))]
        else:
            peaks = np.array(peaks, dtype=int)
            heights = y_for_peaks[peaks]
            order = np.argsort(heights)[::-1]
            peak_inds = list(peaks[order[: max(1, int(n_peaks))]])
            peak_inds.sort()

    if peak_guesses is None and len(peak_inds) > int(n_peaks):
        peak_inds = peak_inds[: int(n_peaks)]

    wn_peaks_argmax = [float(bin_centers[i]) for i in peak_inds]

    # --- determine auto half-window per peak, then union window ---
    half_ws = []
    half_ws_raw = []
    for i in peak_inds:
        hw_raw = _estimate_window_from_fwhm_gap_tolerant(
            bin_centers, y_for_peaks, i, frac=0.5, gap_bins=int(fwhm_gap_bins)
        )
        if hw_raw is None:
            hw_raw = float(min_window_bins * dx)
        half_ws_raw.append(float(hw_raw))

        hw = float(window_scale) * float(hw_raw)
        hw = max(hw, float(min_window_bins * dx))
        half_ws.append(hw)

    lefts = [wn_peaks_argmax[k] - half_ws[k] for k in range(len(peak_inds))]
    rights = [wn_peaks_argmax[k] + half_ws[k] for k in range(len(peak_inds))]
    fit_left = float(min(lefts))
    fit_right = float(max(rights))

    # expand union window a bit so tails aren't cut off
    union_span = max(fit_right - fit_left, dx)
    margin = max(float(fit_margin_bins) * dx, float(fit_margin_frac) * union_span)
    fit_left -= margin
    fit_right += margin

    # clamp to histogram range
    fit_left = max(fit_left, float(bin_centers.min()))
    fit_right = min(fit_right, float(bin_centers.max()))

    # --- unbinned data in fit window ---
    in_win = (x_all >= fit_left) & (x_all <= fit_right)
    x_unb = x_all[in_win]
    if x_unb.size < max(20, 5 * len(peak_inds)):
        raise ValueError(
            f"Too few unbinned points in fit window ({x_unb.size}). "
            f"Consider widening window_scale / fit margins, or check data."
        )

    # -------------------------
    # UNBINNED MLE FIT
    # -------------------------
    if not use_unbinned_mle:
        raise ValueError("use_unbinned_mle=False: this version implements unbinned MLE only.")

    K = len(peak_inds)
    L = float(fit_left)
    R = float(fit_right)
    span = max(R - L, 1e-12)

    # Initial guesses
    mu0 = np.array([float(bin_centers[i]) for i in peak_inds], dtype=float)
    sig0 = np.array([max((2.0 * half_ws_raw[k]) / 2.355, dx) for k in range(K)], dtype=float)

    # weights guess from raw peak heights (no smoothing)
    h = np.array([max(float(y_for_peaks[i]), 1.0) for i in peak_inds], dtype=float)
    w0 = h / np.sum(h)

    b0 = 0.05 if mle_include_uniform_bg else 0.0
    alpha0 = np.log(w0 + 1e-12)
    beta0 = np.log(b0 / (1.0 - b0)) if (mle_include_uniform_bg and (0 < b0 < 1)) else -10.0
    theta0 = np.concatenate([mu0, np.log(sig0), alpha0, np.array([beta0], dtype=float)])

    mu_bounds = [(L, R)] * K
    logsig_bounds = [(np.log(max(1e-6, 0.25 * dx)), np.log(span))] * K
    alpha_bounds = [(None, None)] * K
    beta_bounds = [(None, None)]
    bounds = mu_bounds + logsig_bounds + alpha_bounds + beta_bounds

    def nll(theta):
        mu = theta[0:K]
        logsig = theta[K:2 * K]
        sig = np.exp(logsig)

        alpha = theta[2 * K : 3 * K]
        w = _softmax(alpha)

        if mle_include_uniform_bg:
            beta = theta[-1]
            b = float(_sigmoid(beta))
        else:
            b = 0.0

        pdf = np.zeros_like(x_unb, dtype=float)
        for k in range(K):
            pdf += w[k] * _normal_pdf(x_unb, mu[k], sig[k])

        if mle_include_uniform_bg:
            pdf = (1.0 - b) * pdf + b * (1.0 / span)

        pdf = np.maximum(pdf, 1e-300)
        return -np.sum(np.log(pdf))

    res = minimize(
        nll,
        theta0,
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": int(mle_maxiter)},
    )

    if mle_print:
        print("MLE success:", res.success, "|", res.message)
        print("MLE nll:", res.fun)

    theta_hat = res.x
    mu_hat = theta_hat[0:K]
    sig_hat = np.exp(theta_hat[K:2 * K])
    w_hat = _softmax(theta_hat[2 * K : 3 * K])
    b_hat = float(_sigmoid(theta_hat[-1])) if mle_include_uniform_bg else 0.0

    order = np.argsort(mu_hat)
    mu_hat = mu_hat[order]
    sig_hat = sig_hat[order]
    w_hat = w_hat[order]

    fwhm_hat = 2.355 * sig_hat

    # -------------------------
    # Build curves for plotting in "counts space"
    # -------------------------
    x_model = np.linspace(L, R, 2000)
    Nwin = float(x_unb.size)
    bin_w = float(dx)

    pdf_g = np.zeros_like(x_model)
    for k in range(K):
        pdf_g += w_hat[k] * _normal_pdf(x_model, mu_hat[k], sig_hat[k])

    if mle_include_uniform_bg:
        pdf_total = (1.0 - b_hat) * pdf_g + b_hat * (1.0 / span)
    else:
        pdf_total = pdf_g

    y_total_counts = Nwin * pdf_total * bin_w

    y_components = []
    for k in range(K):
        comp_pdf = ((1.0 - b_hat) * w_hat[k] * _normal_pdf(x_model, mu_hat[k], sig_hat[k])
                    if mle_include_uniform_bg else
                    w_hat[k] * _normal_pdf(x_model, mu_hat[k], sig_hat[k]))
        y_components.append(Nwin * comp_pdf * bin_w)

    y_bg_counts = None
    if mle_include_uniform_bg:
        y_bg_counts = Nwin * (b_hat * (1.0 / span)) * bin_w

    # -------------------------
    # Plot (NO smoothed histogram)
    # -------------------------
    fig, ax = plt.subplots(figsize=(15, 10))

    ax.step(bin_centers, counts, where="mid", color=color, alpha=0.55, label="Histogram (raw)")
    ax.axvspan(fit_left, fit_right, alpha=0.10, label="Fit window")

    ax.plot(x_model, y_total_counts, color="C3", linewidth=2.5, label=f"Unbinned MLE total ({K} Gaussians)")

    for k, yk in enumerate(y_components):
        ax.plot(x_model, yk, linewidth=1.8, alpha=0.9, label=f"Gaussian {k+1}")

    if y_bg_counts is not None:
        ax.plot(x_model, np.full_like(x_model, y_bg_counts), linewidth=1.5, alpha=0.8, label="Uniform background")

    # Argmax markers (from RAW histogram now)
    for k, wn_pk in enumerate(wn_peaks_argmax):
        ax.axvline(
            wn_pk,
            linestyle=":",
            color="k",
            alpha=0.7,
            label=("Argmax peaks (raw)" if k == 0 else None),
        )

    if show_fit_centers:
        for k, x0 in enumerate(mu_hat):
            ax.axvline(x0, linestyle="--", color="k", label=("MLE centers" if k == 0 else None))

    ax.set_title(title, fontweight="bold", fontsize="x-large")
    ax.set_xlabel(xlabel, fontsize="x-large", fontweight="bold")
    ax.set_ylabel(ylabel, fontsize="x-large", fontweight="bold")
    ax.legend()

    if plot_window_scale is None:
        ax.set_xlim(xmin, xmax)
    else:
        half_w_max = max(half_ws) if len(half_ws) else float(min_window_bins * dx)
        center_cluster = 0.5 * (min(wn_peaks_argmax) + max(wn_peaks_argmax))
        plot_half_w = float(plot_window_scale) * half_w_max
        x_left = max(center_cluster - plot_half_w, float(bin_centers.min()))
        x_right = min(center_cluster + plot_half_w, float(bin_centers.max()))
        ax.set_xlim(xmin, xmax)

    fig.tight_layout()
    plt.show()

    return {
        "wn_col": wn_col,
        "wn_peaks_argmax_raw": wn_peaks_argmax,
        "fit_window": (fit_left, fit_right),
        "mle_success": bool(res.success),
        "mle_message": str(res.message),
        "mle_nll": float(res.fun),
        "mle_centers": mu_hat.tolist(),
        "mle_sigmas": sig_hat.tolist(),
        "mle_fwhms": fwhm_hat.tolist(),
        "mle_weights": w_hat.tolist(),
        "mle_bg_fraction": float(b_hat),
        "bin_centers": bin_centers,
        "counts": counts,
        "x_unbinned_in_window": x_unb,
        "x_model": x_model,
        "y_total_counts_model": y_total_counts,
        "y_component_counts_models": y_components,
        "y_bg_counts_model": float(y_bg_counts) if y_bg_counts is not None else None,
    }

In [ ]:
result = plot_hist_spec_multigauss_auto_window(
    file=r"C:\Users\j_hac\OneDrive\Desktop\Exotic Molecules lab\Sulfur\Final scans data\scan_2025_12_12_15_48_19.csv",
    bins=200,
    title=r"Grating Ti:sa Full ${}^{4}P_{J'}$ Scan",
    xmin = 1.26237e4,
    xmax=1.26273e4,
    wn_col="wn_3",

    # Peak finding (raw histogram)
    n_peaks=6,
    peak_min_distance_bins=15,
    peak_prominence=None,      # let it choose ~10% of max by default

    # Windowing / robustness to zero-bin gaps
    window_scale=3.0,
    min_window_bins=12,
    fwhm_gap_bins=3,            # tolerate small zero gaps inside peaks

    # Plotting
    plot_window_scale=None,     # show full spectrum

    # Fit
    use_unbinned_mle=True,
    mle_include_uniform_bg=True,
    show_fit_centers=False,     # keep only argmax markers
    mle_print=True,             # prints optimizer success/message
)

print("\nArgmax (raw histogram) peak positions:")
for i, wn in enumerate(result["wn_peaks_argmax_raw"], 1):
    print(f"  Peak {i}: {wn:.6f} cm^-1")

print("\nUnbinned MLE fitted centers (mu):")
for i, wn in enumerate(result["mle_centers"], 1):
    print(f"  Peak {i}: {wn:.6f} cm^-1")

print("\nFitted FWHMs (cm^-1):")
for i, fwhm in enumerate(result["mle_fwhms"], 1):
    print(f"  Peak {i}: {fwhm:.6f}")

print("\nEstimated uniform background fraction b:", result["mle_bg_fraction"])
print("Fit window:", result["fit_window"])

In [13]:
# Calculate wavenumber/wavelength for atomic excitation of given ion/HG/potential/geometry 

def beam_corrected_transition(
    measured_wavenumber_cm1: float,
    ion_charge_state: float,
    ion_mass_u: float,
    potential_V: float,
    fundamental_multiple: float = 1.0,
    geometry: str = "co",
    theta_deg: Optional[float] = None,
):
    """
    Convert a measured LAB wavenumber (typically of the fundamental) into the
    ion-rest-frame transition wavenumber and wavelength, accounting for:
      - harmonic generation: fundamental_multiple (e.g., 2 for SHG)
      - beam acceleration through potential_V (relativistic)
      - Doppler shift for co-/counter-propagating or arbitrary angle

    Parameters
    ----------
    measured_wavenumber_cm1 : float
        Wavenumber of the light you measured in the lab (cm^-1).
        Often this is the fundamental wavenumber from a wavemeter.
    ion_charge_state : float
        Charge state z (e.g., +1 for S+, +2 for S2+, etc.).
    ion_mass_u : float
        Ion mass in atomic mass units (u). For 32S, use ~31.972 u (or 32 as approx).
    potential_V : float
        Acceleration potential in volts (V). For 10 kV, use 10_000.
    fundamental_multiple : float
        Multiply measured light frequency/wavenumber by this to get the photon
        interacting with the ion (e.g., 2 for SHG, 3 for THG).
    geometry : str
        "co" for co-propagating, "counter" for counter-propagating,
        "angle" for arbitrary angle theta_deg between laser k-vector and ion velocity.
    theta_deg : float | None
        Angle in degrees between laser propagation direction and ion velocity.
        theta=0 => co-propagating; theta=180 => counter-propagating.

    Returns
    -------
    result : dict
        Contains:
          - wn_lab_photon_cm1 : wavenumber of the photon at the ions in the lab frame
          - wn_ion_cm1        : ion-rest-frame transition wavenumber (cm^-1)
          - lambda_ion_nm     : ion-rest-frame transition wavelength (vacuum, nm)
          - beta, gamma, v_ms : beam kinematics
          - cos_theta         : used projection for Doppler
    """
    # Physical constants (exact where defined)
    C = 299_792_458.0               # m/s
    E_CHARGE = 1.602_176_634e-19    # C
    U = 1.66053906660e-27        # kg (atomic mass unit)

    # --- beam kinematics (relativistic) ---
    z = ion_charge_state
    m = ion_mass_u * U
    qV_J = (z * E_CHARGE) * potential_V  # Joules of kinetic energy (nonrel approx)
    mc2 = m * C**2

    # Relativistic gamma from kinetic energy: K = (gamma-1) mc^2
    gamma = 1.0 + (qV_J / mc2)
    beta = np.sqrt(1.0 - 1.0 / gamma**2)
    v_ms = beta * C

    # --- geometry ---
    g = geometry.lower().strip()
    if g in ("co", "coprop", "co-prop", "colinear", "co-linear"):
        cos_theta = 1.0
    elif g in ("counter", "counterprop", "counter-prop", "anti", "antiparallel"):
        cos_theta = -1.0
    elif g in ("angle", "theta"):
        if theta_deg is None:
            raise ValueError('theta_deg must be provided when geometry="angle".')
        cos_theta = float(np.cos(np.deg2rad(theta_deg)))
    else:
        raise ValueError('geometry must be "co", "counter", or "angle".')

    # --- wavenumber of the photon interacting with the ions in the LAB frame ---
    wn_lab_photon_cm1 = float(measured_wavenumber_cm1) * float(fundamental_multiple)

    # --- Doppler transform from LAB to ION REST FRAME ---
    # Ion-frame frequency: nu' = gamma * (1 - beta*cosθ) * nu
    # Same scaling for wavenumber (vacuum): wn' = gamma * (1 - beta*cosθ) * wn
    wn_ion_cm1 = wn_lab_photon_cm1 * gamma * (1.0 - beta * cos_theta)

    # --- convert to vacuum wavelength ---
    lambda_ion_nm = 1e7 / wn_ion_cm1

    return {
        "wn_lab_photon_cm1": wn_lab_photon_cm1,
        "wn_ion_cm1": wn_ion_cm1,
        "lambda_ion_nm": lambda_ion_nm,
        "beta": float(beta),
        "gamma": float(gamma),
        "v_ms": float(v_ms),
        "cos_theta": float(cos_theta)}

In [14]:
#Example in S

res = beam_corrected_transition(
    measured_wavenumber_cm1=12625.187,
    ion_charge_state=1,
    ion_mass_u=32.0,
    potential_V=10_000,
    fundamental_multiple=2,   # SHG
    geometry="co"
)

print("Wavenumber for atomic transition is: ", res["wn_ion_cm1"], "1/cm")
print("Wavelength for atomic transition is: ", res["lambda_ion_nm"], "nm")

Wavenumber for atomic transition is:  25229.699267536405 1/cm
Wavelength for atomic transition is:  396.3582718113178 nm


In [11]:
#Backwards program finds wavenumber to scan for target transition

import numpy as np
from typing import Optional

# Physical constants
C = 299_792_458.0               # m/s
E_CHARGE = 1.602_176_634e-19    # C
U = 1.660_539_066_60e-27        # kg (atomic mass unit)

def _beam_beta_gamma(ion_charge_state: float, ion_mass_u: float, potential_V: float):
    """Return (beta, gamma, v_ms) for an ion accelerated through potential_V."""
    z = float(ion_charge_state)
    m = float(ion_mass_u) * U
    V = float(potential_V)

    qV_J = z * E_CHARGE * V
    mc2 = m * C**2

    gamma = 1.0 + (qV_J / mc2)                 # K = (gamma-1)mc^2
    beta = np.sqrt(1.0 - 1.0 / gamma**2)
    v_ms = beta * C
    return float(beta), float(gamma), float(v_ms)

def _cos_theta_from_geometry(geometry: str, theta_deg: Optional[float]):
    g = geometry.lower().strip()
    if g in ("co", "coprop", "co-prop", "colinear", "co-linear"):
        return 1.0
    if g in ("counter", "counterprop", "counter-prop", "anti", "antiparallel"):
        return -1.0
    if g in ("angle", "theta"):
        if theta_deg is None:
            raise ValueError('theta_deg must be provided when geometry="angle".')
        return float(np.cos(np.deg2rad(theta_deg)))
    raise ValueError('geometry must be "co", "counter", or "angle".')

def beam_corrected_transition(
    measured_wavenumber_cm1: float,
    ion_charge_state: float,
    ion_mass_u: float,
    potential_V: float,
    fundamental_multiple: float = 1.0,
    geometry: str = "co",
    theta_deg: Optional[float] = None,
):
    """
    LAB measured fundamental wavenumber -> ion-rest-frame transition wavenumber & wavelength.
    """
    beta, gamma, v_ms = _beam_beta_gamma(ion_charge_state, ion_mass_u, potential_V)
    cos_theta = _cos_theta_from_geometry(geometry, theta_deg)

    wn_lab_photon_cm1 = float(measured_wavenumber_cm1) * float(fundamental_multiple)

    # nu' = gamma(1 - beta cosθ) nu  -> same factor for wavenumber
    doppler_factor = gamma * (1.0 - beta * cos_theta)
    wn_ion_cm1 = wn_lab_photon_cm1 * doppler_factor
    lambda_ion_nm = 1e7 / wn_ion_cm1

    return {
        "wn_lab_photon_cm1": wn_lab_photon_cm1,
        "wn_ion_cm1": wn_ion_cm1,
        "lambda_ion_nm": lambda_ion_nm,
        "beta": beta,
        "gamma": gamma,
        "v_ms": v_ms,
        "cos_theta": cos_theta,
        "doppler_factor": doppler_factor,
    }

def required_fundamental_wavenumber(
    target_transition_ion_cm1: float,
    ion_charge_state: float,
    ion_mass_u: float,
    potential_V: float,
    fundamental_multiple: float = 1.0,
    geometry: str = "co",
    theta_deg: Optional[float] = None,
):
    """
    Ion-rest-frame target transition wavenumber -> required LAB fundamental wavenumber & wavelength.

    Returns fundamental LAB wavenumber (cm^-1) to dial on wavemeter (before harmonic multiple),
    plus the corresponding fundamental LAB wavelength (vacuum, nm), and the LAB photon wavelength.
    """
    beta, gamma, v_ms = _beam_beta_gamma(ion_charge_state, ion_mass_u, potential_V)
    cos_theta = _cos_theta_from_geometry(geometry, theta_deg)

    doppler_factor = gamma * (1.0 - beta * cos_theta)  # ion = lab_photon * factor

    M = float(fundamental_multiple)
    if M <= 0:
        raise ValueError("fundamental_multiple must be > 0.")

    # Invert: lab_photon = ion / doppler_factor; fundamental = lab_photon / M
    wn_lab_photon_cm1 = float(target_transition_ion_cm1) / doppler_factor
    wn_fund_cm1 = wn_lab_photon_cm1 / M

    # Convert to vacuum wavelengths (nm)
    lambda_fund_nm = 1e7 / wn_fund_cm1
    lambda_lab_photon_nm = 1e7 / wn_lab_photon_cm1

    return {
        "wn_fund_cm1": wn_fund_cm1,                      # what wavemeter sees (fundamental)
        "lambda_fund_nm": lambda_fund_nm,                # fundamental vacuum wavelength
        "wn_lab_photon_cm1": wn_lab_photon_cm1,          # after harmonic multiple
        "lambda_lab_photon_nm": lambda_lab_photon_nm,    # photon vacuum wavelength in lab
        "beta": beta,
        "gamma": gamma,
        "v_ms": v_ms,
        "cos_theta": cos_theta,
        "doppler_factor": doppler_factor,
        "fundamental_multiple": M,
    }


In [12]:
#Example in S

res_inv = required_fundamental_wavenumber(
    target_transition_ion_cm1=25229.0,
    ion_charge_state=1,
    ion_mass_u=32.0,
    potential_V=10_000,
    fundamental_multiple=2,   # SHG
    geometry="co"
)

print("Dial this (fundamental) wavenumber:", res_inv["wn_fund_cm1"], "cm^-1")
print("Fundamental wavelength:", res_inv["lambda_fund_nm"], "nm (vac)")
print("Doubled photon wavelength:", res_inv["lambda_lab_photon_nm"], "nm (vac)")


Dial this (fundamental) wavenumber: 12624.837079720868 cm^-1
Fundamental wavelength: 792.0894294994813 nm (vac)
Doubled photon wavelength: 396.04471474974065 nm (vac)
